<a href="https://colab.research.google.com/github/Tolureis/CMPE-460-Deep-Learning/blob/main/MNIST_1D_Performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""8_1_MNIST_1D_Performance.py
Updated with completed TODOs.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ===========================
# 1. DATA PREPARATION
# ===========================
batch_size = 64

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ===========================
# 2. MODEL DEFINITION
# ===========================

D_i = 40    # Input dimensions (1D flattened features)
D_k = 100   # Hidden dimensions
D_o = 10    # Output dimensions

class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(D_i, D_k),
            nn.ReLU(inplace=True),
            nn.Linear(D_k, D_k),
            nn.ReLU(inplace=True),
            nn.Linear(D_k, D_o)
        )
        self.net.apply(self.weights_init)

    def weights_init(self, layer_in):
        if isinstance(layer_in, nn.Linear):
            nn.init.kaiming_normal_(layer_in.weight, nonlinearity='relu')
            if layer_in.bias is not None:
                nn.init.zeros_(layer_in.bias)

    def forward(self, x):
        return self.net(x)

model = MLP()

# ===========================
# 3. TRAINING SETUP
# ===========================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ===========================
# 4. TRAINING LOOP
# ===========================
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images = images.view(images.size(0), -1)[:, :D_i]  # Use first 40 features for demo
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

# ===========================
# 5. TESTING LOOP
# ===========================

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.view(images.size(0), -1)[:, :D_i]
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 488kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.52MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.98MB/s]


Epoch [1/5], Loss: 2.3124
Epoch [2/5], Loss: 2.3037
Epoch [3/5], Loss: 2.3011
Epoch [4/5], Loss: 2.2998
Epoch [5/5], Loss: 2.2998
Test Accuracy: 11.48%
